# Trendlines with Breaks [LuxAlgo] 概念實作與多標的策略回測

本 notebook 依照期中作業要求，實作一個參考 **Trendlines with Breaks [LuxAlgo]** 概念的技術指標策略。原指標重點是使用 pivot point 建立趨勢線，並標示價格突破趨勢線的訊號。LuxAlgo 官方說明指出，該指標以 pivot-based trendlines 尋找 breakout，並可使用 ATR、標準差或線性迴歸作為趨勢線斜率計算方式。

參考來源：

- LuxAlgo indicator library: https://www.luxalgo.com/library/indicator/trendlines-with-breaks
- TradingView open-source indicator page: https://www.tradingview.com/script/IYL88A1N-Trendlines-with-Breaks-LuxAlgo/

本作業不是直接複製 Pine Script，而是以 Python 重新建立可回測版本。為避免未來函數問題，pivot high/low 必須等右側 `length` 根 K 線完成後才確認，因此策略訊號會自然延遲。

## 1. 套件與研究設定

In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import os
from itertools import product

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

START_DATE = "2018-01-01"
END_DATE = "2026-05-06"
CACHE_DIR = "price_cache"

TICKERS = ["TSM", "AAPL", "NVDA", "MSFT", "AMD", "SPY", "QQQ"]
MAIN_TICKER = "TSM"

TRANSACTION_COST = 0.001425
STOP_LOSS = 0.12
TAKE_PROFIT = None

## 2. 資料抓取與預處理

使用 `yfinance` 抓取日資料，並建立本地快取 CSV，避免重複下載造成 Yahoo Finance 限流。回測使用調整後收盤價 `Adj Close` 作為交易價格，以反映拆股與股利調整。

In [ ]:
def download_price_data(ticker, start=START_DATE, end=END_DATE, use_cache=True):
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(CACHE_DIR, f"{ticker}_{start}_{end}.csv")

    if use_cache and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        print(f"使用快取資料：{ticker}")
    else:
        df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
        if df.empty:
            raise ValueError(f"無法下載 {ticker}，可能是 Yahoo Finance 暫時限流，請稍後重試。")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.reset_index()
        df.to_csv(cache_path, index=False)
        print(f"已下載並建立快取：{ticker}")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates("Date").set_index("Date")
    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    df["Price"] = df[price_col]
    df = df.dropna(subset=["Open", "High", "Low", "Close", "Price", "Volume"])
    df["Return"] = df["Price"].pct_change()
    return df.dropna(subset=["Return"])

sample = download_price_data(MAIN_TICKER)
display(sample.head())
display(sample.tail())
print(f"{MAIN_TICKER} 資料期間：{sample.index.min().date()} 到 {sample.index.max().date()}，共 {len(sample):,} 筆")

## 3. 指標邏輯說明

本 notebook 將 Trendlines with Breaks 的概念拆成三步：

1. **Pivot 偵測**：若某日 high 是左右各 `length` 日內最高，則為 pivot high；若 low 是左右各 `length` 日內最低，則為 pivot low。因為需要右側資料確認，所以 pivot 會在 `length` 日後才可用。
2. **趨勢線延伸**：pivot high 形成上方壓力線，pivot low 形成下方支撐線。斜率可用 ATR、標準差或線性迴歸估計。
3. **Breakout 訊號**：價格突破上方趨勢線視為多方突破；跌破下方趨勢線視為弱勢或出場訊號。

這裡的實作重點是回測可信度，因此所有 pivot 都只在確認後才加入訊號計算。

In [ ]:
def add_atr(df, window=14):
    out = df.copy()
    prev_close = out["Close"].shift(1)
    tr = pd.concat([
        out["High"] - out["Low"],
        (out["High"] - prev_close).abs(),
        (out["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    out["ATR"] = tr.rolling(window).mean()
    return out

def rolling_linreg_slope(series, window):
    x = np.arange(window)
    x_centered = x - x.mean()
    denominator = (x_centered ** 2).sum()

    def slope(values):
        if np.isnan(values).any():
            return np.nan
        y = values - values.mean()
        return float((x_centered * y).sum() / denominator)

    return series.rolling(window).apply(slope, raw=True)

def slope_series(df, length, method="atr", mult=1.0):
    if method == "atr":
        return df["ATR"] / length * mult
    if method == "stdev":
        return df["Price"].rolling(length).std() / length * mult
    if method == "linreg":
        return rolling_linreg_slope(df["Price"], length).abs() * mult
    raise ValueError("method must be atr, stdev, or linreg")

def add_trendline_breaks(df, length=14, slope_method="atr", slope_mult=1.0):
    out = add_atr(df, window=max(14, length)).copy()
    slope = slope_series(out, length, slope_method, slope_mult).fillna(0)
    n = len(out)

    pivot_high_confirmed = np.zeros(n, dtype=bool)
    pivot_low_confirmed = np.zeros(n, dtype=bool)

    highs = out["High"].to_numpy()
    lows = out["Low"].to_numpy()
    for i in range(length, n - length):
        high_window = highs[i - length:i + length + 1]
        low_window = lows[i - length:i + length + 1]
        if highs[i] == np.nanmax(high_window):
            pivot_high_confirmed[i + length] = True
        if lows[i] == np.nanmin(low_window):
            pivot_low_confirmed[i + length] = True

    upper = np.full(n, np.nan)
    lower = np.full(n, np.nan)
    current_upper = np.nan
    current_lower = np.nan
    current_upper_slope = 0.0
    current_lower_slope = 0.0

    for i in range(n):
        if pivot_high_confirmed[i]:
            pivot_index = i - length
            current_upper = highs[pivot_index] - slope.iloc[i] * length
            current_upper_slope = slope.iloc[i]
        elif not np.isnan(current_upper):
            current_upper -= current_upper_slope

        if pivot_low_confirmed[i]:
            pivot_index = i - length
            current_lower = lows[pivot_index] + slope.iloc[i] * length
            current_lower_slope = slope.iloc[i]
        elif not np.isnan(current_lower):
            current_lower += current_lower_slope

        upper[i] = current_upper
        lower[i] = current_lower

    out["Upper_Trendline"] = upper
    out["Lower_Trendline"] = lower
    out["Pivot_High_Confirmed"] = pivot_high_confirmed
    out["Pivot_Low_Confirmed"] = pivot_low_confirmed
    out["Upper_Break"] = (out["Close"] > out["Upper_Trendline"]) & (out["Close"].shift(1) <= out["Upper_Trendline"].shift(1))
    out["Lower_Break"] = (out["Close"] < out["Lower_Trendline"]) & (out["Close"].shift(1) >= out["Lower_Trendline"].shift(1))
    return out

## 4. ????

? notebook ?? 200 ?????????? trendline breakout ????????????????????

1. **Basic Stop**???????????????????????? 12% ???
2. **Breakeven Stop**??????? Basic Stop????????????? 5%??????????????????????

?????????????????????????????????????????????? long-only??????????????????

In [ ]:
def backtest_breakout_strategy(
    df,
    stop_mode="basic",
    stop_loss=STOP_LOSS,
    breakeven_trigger=0.05,
    transaction_cost=TRANSACTION_COST,
):
    out = df.copy()

    units = 0
    entry_price = np.nan
    highest_price_since_entry = np.nan
    stop_price = np.nan
    breakeven_activated = False

    positions = []
    actions = []
    stop_prices = []
    breakeven_flags = []

    for i in range(len(out)):
        if i == 0:
            positions.append(0)
            actions.append("Hold")
            stop_prices.append(np.nan)
            breakeven_flags.append(False)
            continue

        price = out["Price"].iloc[i]
        upper_break = bool(out["Upper_Break"].iloc[i - 1])
        lower_break = bool(out["Lower_Break"].iloc[i - 1])
        action = "Hold"

        if units == 0:
            if upper_break:
                units = 1
                entry_price = price
                highest_price_since_entry = price
                stop_price = entry_price * (1 - stop_loss)
                breakeven_activated = False
                action = "Buy"
        else:
            highest_price_since_entry = max(highest_price_since_entry, price)

            if stop_mode == "breakeven" and not breakeven_activated:
                if highest_price_since_entry >= entry_price * (1 + breakeven_trigger):
                    stop_price = entry_price
                    breakeven_activated = True

            stopped = price <= stop_price
            if lower_break or stopped:
                if stopped and breakeven_activated:
                    action = "Breakeven Stop"
                elif stopped:
                    action = "Stop Loss"
                else:
                    action = "Sell"
                units = 0
                entry_price = np.nan
                highest_price_since_entry = np.nan
                stop_price = np.nan
                breakeven_activated = False

        positions.append(units)
        actions.append(action)
        stop_prices.append(stop_price)
        breakeven_flags.append(breakeven_activated)

    out["Position"] = positions
    out["Action"] = actions
    out["Stop_Price"] = stop_prices
    out["Breakeven_Activated"] = breakeven_flags
    out["Trade"] = out["Position"].diff().abs().fillna(out["Position"].abs())
    out["Strategy_Return_Gross"] = out["Position"].shift(1).fillna(0) * out["Return"]
    out["Transaction_Cost"] = out["Trade"] * transaction_cost
    out["Strategy_Return"] = out["Strategy_Return_Gross"] - out["Transaction_Cost"]
    out["Strategy_Cum"] = (1 + out["Strategy_Return"]).cumprod()
    out["BuyHold_Cum"] = (1 + out["Return"]).cumprod()
    return out

def max_drawdown(cumulative_return):
    running_max = cumulative_return.cummax()
    drawdown = cumulative_return / running_max - 1
    return drawdown.min()

def performance_metrics(df, return_col="Strategy_Return", cumulative_col="Strategy_Cum"):
    daily_ret = df[return_col].dropna()
    total_return = df[cumulative_col].iloc[-1] - 1
    years = len(daily_ret) / 252
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_vol = daily_ret.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol != 0 else np.nan
    return {
        "Total Return": total_return,
        "Annual Return": annual_return,
        "Annual Volatility": annual_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_drawdown(df[cumulative_col]),
        "Win Rate": (daily_ret > 0).mean(),
    }

demo = add_trendline_breaks(sample, length=14, slope_method="atr", slope_mult=1.0)
demo_basic = backtest_breakout_strategy(demo, stop_mode="basic")
demo_breakeven = backtest_breakout_strategy(demo, stop_mode="breakeven")
display(demo_breakeven[["Price", "Upper_Trendline", "Lower_Trendline", "Upper_Break", "Lower_Break", "Position", "Stop_Price", "Breakeven_Activated", "Action"]].tail(15))

## 5. ???????

?????? pivot ??????????????????????????????? breakout????? 200 ??????

?????????? **Total Return ? Sharpe Ratio ?? Buy and Hold** ???????????????????????

In [ ]:
param_grid = list(product(
    [8, 14, 21],
    ["atr", "stdev", "linreg"],
    [0.5, 1.0, 1.5],
    ["basic", "breakeven"],
))

all_results = []
best_backtests = {}

for ticker in TICKERS:
    try:
        price_data = download_price_data(ticker)
    except Exception as err:
        print(f"?? {ticker}?{err}")
        continue

    buyhold_tmp = price_data.copy()
    buyhold_tmp["BuyHold_Cum"] = (1 + buyhold_tmp["Return"]).cumprod()
    buyhold_metrics = performance_metrics(
        buyhold_tmp.assign(BuyHold_Return=buyhold_tmp["Return"]),
        return_col="BuyHold_Return",
        cumulative_col="BuyHold_Cum",
    )

    ticker_best = None
    ticker_best_score = -np.inf

    for length, slope_method, slope_mult, stop_mode in param_grid:
        indicator = add_trendline_breaks(price_data, length=length, slope_method=slope_method, slope_mult=slope_mult)
        bt = backtest_breakout_strategy(indicator, stop_mode=stop_mode)
        metrics = performance_metrics(bt)

        row = {
            "Ticker": ticker,
            "Length": length,
            "Slope Method": slope_method,
            "Slope Mult": slope_mult,
            "Stop Mode": stop_mode,
            "Trades": int(bt["Trade"].sum()),
            "Buy Actions": int((bt["Action"] == "Buy").sum()),
            "Trendline Sells": int((bt["Action"] == "Sell").sum()),
            "Stop Losses": int((bt["Action"] == "Stop Loss").sum()),
            "Breakeven Stops": int((bt["Action"] == "Breakeven Stop").sum()),
            "BH Total Return": buyhold_metrics["Total Return"],
            "BH Sharpe": buyhold_metrics["Sharpe"],
            **metrics,
        }
        row["Return Beats BH"] = row["Total Return"] > row["BH Total Return"]
        row["Sharpe Beats BH"] = row["Sharpe"] > row["BH Sharpe"]
        row["Both Beat BH"] = row["Return Beats BH"] and row["Sharpe Beats BH"]
        all_results.append(row)

        score = row["Sharpe"] - row["BH Sharpe"] + 0.25 * (row["Total Return"] - row["BH Total Return"])
        if row["Trades"] >= 2 and score > ticker_best_score:
            ticker_best_score = score
            ticker_best = (row, bt)

    if ticker_best is not None:
        best_backtests[ticker] = ticker_best

results = pd.DataFrame(all_results)
display(results.head())
print(f"????????{len(results):,}")

## 6. 找出每個標的最佳策略

In [ ]:
best_by_ticker = []
for ticker, (row, bt) in best_backtests.items():
    best_by_ticker.append(row)

best_table = pd.DataFrame(best_by_ticker).sort_values(["Sharpe Beats BH", "Return Beats BH", "Sharpe"], ascending=False)
display(
    best_table.style
    .format("{:.2%}", subset=["Total Return", "Annual Return", "Annual Volatility", "Max Drawdown", "Win Rate", "BH Total Return"])
    .format("{:.2f}", subset=["Sharpe", "BH Sharpe"])
)

beat_summary = results.groupby("Ticker")[["Return Beats BH", "Sharpe Beats BH", "Both Beat BH"]].sum().astype(int)
beat_summary["Tested Combinations"] = results.groupby("Ticker").size()
display(beat_summary)

## 7. 圖表：最佳策略 vs Buy and Hold

In [ ]:
plt.figure(figsize=(15, 7))
for ticker, (row, bt) in best_backtests.items():
    plt.plot(bt.index, bt["Strategy_Cum"], label=f"{ticker} Strategy")
plt.title("Best Trendline Break Strategy by Ticker")
plt.ylabel("Growth of $1")
plt.xlabel("Date")
plt.legend()
plt.show()

comparison = best_table.set_index("Ticker")[["Total Return", "BH Total Return", "Sharpe", "BH Sharpe"]]
comparison[["Total Return", "BH Total Return"]].plot(kind="bar", figsize=(12, 5), title="Best Strategy Return vs Buy and Hold")
plt.ylabel("Total Return")
plt.xticks(rotation=0)
plt.show()

comparison[["Sharpe", "BH Sharpe"]].plot(kind="bar", figsize=(12, 5), title="Best Strategy Sharpe vs Buy and Hold")
plt.ylabel("Sharpe Ratio")
plt.xticks(rotation=0)
plt.show()

## 8. 單一標的指標圖：Trendlines with Breaks

In [ ]:
def plot_trendline_breaks(bt, ticker):
    plot_df = bt.tail(500).copy()
    buys = plot_df[plot_df["Action"] == "Buy"]
    sells = plot_df[plot_df["Action"].isin(["Sell", "Stop Loss"])]

    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"height_ratios": [3, 1, 1.5]})
    axes[0].plot(plot_df.index, plot_df["Price"], label="Adjusted Price", color="#1f77b4")
    axes[0].plot(plot_df.index, plot_df["Upper_Trendline"], label="Upper Trendline", color="#d62728", linestyle="--")
    axes[0].plot(plot_df.index, plot_df["Lower_Trendline"], label="Lower Trendline", color="#2ca02c", linestyle="--")
    axes[0].scatter(buys.index, buys["Price"], marker="^", color="green", s=70, label="Buy")
    axes[0].scatter(sells.index, sells["Price"], marker="v", color="red", s=70, label="Sell / Stop")
    axes[0].set_title(f"{ticker} Trendlines with Breaks Strategy Signals")
    axes[0].set_ylabel("Price")
    axes[0].legend(loc="upper left")

    axes[1].step(plot_df.index, plot_df["Position"], where="post", color="#9467bd")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].set_ylabel("Position")
    axes[1].set_title("Position")

    axes[2].plot(plot_df.index, plot_df["Strategy_Cum"], label="Strategy", color="#17becf")
    axes[2].plot(plot_df.index, plot_df["BuyHold_Cum"], label="Buy and Hold", color="#7f7f7f")
    axes[2].set_ylabel("Growth of $1")
    axes[2].set_title("Cumulative Return")
    axes[2].legend(loc="upper left")
    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

main_row, main_bt = best_backtests.get(MAIN_TICKER, next(iter(best_backtests.values())))
plot_trendline_breaks(main_bt, MAIN_TICKER)

## 9. 結果解釋與討論

Trendline breakout 策略的核心邏輯是：當價格突破由 pivot high 延伸出的上方趨勢線時，代表原本的下降壓力可能被打破，因此進場做多；當價格跌破由 pivot low 延伸出的下方趨勢線時，代表支撐失效，因此出場。

回測結果需要同時觀察報酬率與 Sharpe Ratio。若策略報酬率高於 buy and hold，但 Sharpe Ratio 較低，代表策略可能只是承擔較高波動換取報酬；若 Sharpe Ratio 高於 buy and hold，但總報酬較低，則代表策略可能有較佳風險控制，但資金長期曝險不足。

多標的搜尋的目的不是保證找到永久有效的參數，而是觀察策略是否只對單一股票有效。如果只有少數標的能打敗 buy and hold，表示策略可能依賴標的特性；如果多數標的都能在 Sharpe 或最大回撤上改善，則代表策略具有較好的穩定性。

## 10. 生成式 AI 應用與反思

本作業使用生成式 AI 協助將 TradingView 指標概念轉換為 Python 回測流程。為了提高程式品質，我先要求 AI 查清楚指標的核心特徵，例如 pivot-based trendlines、breakout signals，以及 ATR、Stdev、Linreg 三種斜率方法，再要求它避免未來函數問題。

實作過程中特別需要注意 pivot 指標的延遲。若直接用當天 pivot high/low 產生訊號，回測會偷看到未來價格，導致結果過度樂觀。因此 notebook 中設定 pivot 必須等待右側 `length` 根 K 線後才確認，交易也使用前一日訊號在下一日執行。

未來可以進一步加入 walk-forward validation，把前半段資料用於尋找參數，後半段資料用於驗證，降低資料探勘造成的過度配適風險。

## 11. 結論

本 notebook 完成 Trendlines with Breaks 概念的 Python 實作，包含資料抓取、資料處理、pivot-based trendline 計算、breakout 交易策略、多標的參數搜尋、圖表視覺化與績效分析。

策略是否優於 buy and hold，取決於標的特性、參數設定與市場環境。若回測表中出現 return 或 Sharpe Ratio 勝過 buy and hold 的組合，代表此指標具有進一步研究價值；但若要實際交易，仍需要樣本外測試、滑價估計與更完整的風險控管。